# Movie time point classification (Fig. 2, Supp. Fig. S3)

In [ ]:
import os
from glob import glob
import io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import scipy.stats as stats
from PIL import Image
import neuroboros as nb

root = "/dartfs/rc/lab/H/HaxbyLab/yuqi/monkey_kingdom_data/"

## Monkey (Fig. 2)

In [ ]:
DATA_ROOT = "/dartfs/rc/lab/H/HaxbyLab/monkey_kingdom/feilong/data/monkeys/nb-2s/mkavg-ico32"
folders = sorted(glob(f"{DATA_ROOT}/*"))
sids = [os.path.basename(_) for _ in folders]
sids

In [ ]:
# For each subject, pick the number of PCs (npc) that maximizes nested
# leave-two-out ("train") accuracy, then read off the held-out ("test")
# accuracy at that npc -- this is the number reported in the paper.
def npc_selected_accuracy(sid_list, train_dir, train_pattern, test_dir, test_pattern):
    hyper_max_acc = []
    for sid in sid_list:
        accs = [
            np.load(f"{root}{train_dir}/{train_pattern.format(exclude_sid=exclude_sid, sid=sid)}")
            for exclude_sid in sid_list if exclude_sid != sid
        ]
        avg = np.mean(accs, axis=0)
        test = np.load(f"{root}{test_dir}/{test_pattern.format(sid=sid)}")
        max_idx = np.argmax(avg[:30])
        hyper_max_acc.append(test[max_idx])
    return np.array(hyper_max_acc)


hyper_max_acc = npc_selected_accuracy(
    sids,
    "classification/hyperaligned_train",
    "{exclude_sid}_all_ridge_with_mask_unweighted_exclude_{sid}.npy",
    "classification/hyperaligned",
    "{sid}_all_ridge_with_mask_unweighted_latest.npy",
)
anatomical_max_acc = npc_selected_accuracy(
    sids,
    "classification/anatomical_train",
    "{exclude_sid}_with_mask_exclude_{sid}.npy",
    "classification/anatomical",
    "{sid}_with_mask_latest.npy",
)

In [ ]:
# Panel c: searchlight-average classification accuracy, computed directly
# from the per-subject searchlight results (validated against on-disk
# outputs in classify_timepoints_searchlight_monkey.py)
mask = nb.mask('lr', 'mkavg-ico32')


def searchlight_avg_map(out_tag):
    accs = [
        np.load(f"{root}classification/searchlight_analysis/{sid}_{out_tag}_timepoint_latest.npy")
        for sid in sids
    ]
    avg = np.mean(accs, axis=0)
    full = np.full(len(mask), np.nan)
    full[mask] = avg
    return full


anat_sl = searchlight_avg_map("anatomical_ridge_with_mask")
hyper_sl = searchlight_avg_map("hyperalign_to_template_ridge_with_mask_unweighted")

vmin, vmax = 0, 0.15
cmap = 'magma'
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
im1 = nb.plot_mebrains(anat_sl, vmax=vmax, vmin=vmin, cmap=cmap, colorbar=False, title="Anatomically-aligned")
im3 = nb.plot_mebrains(hyper_sl, vmax=vmax, vmin=vmin, cmap=cmap, colorbar=False, title="Hyperaligned")

fig_cb, ax_cb = plt.subplots(figsize=(11.3, 0.15), dpi=300)
fig_cb.subplots_adjust(0, 0, 1, 1)
mpl.colorbar.ColorbarBase(ax_cb, cmap=cmap, norm=norm, orientation='horizontal', ticks=[0, .05, .10, .15])
buf = io.BytesIO()
fig_cb.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
plt.close(fig_cb)
buf.seek(0)
cb_h = Image.open(buf).convert('RGBA')

top = nb.Image.hstack([im1, im3], padding=12)
panel_c = nb.Image.vstack([top, cb_h], padding=8)
png_bytes = panel_c._repr_png_()
panel_pil = Image.open(io.BytesIO(png_bytes)).convert('RGBA')
white_bg = Image.new("RGBA", panel_pil.size, "WHITE")
final_panel_c = Image.alpha_composite(white_bg, panel_pil).convert('RGB')

In [ ]:
# Panels a+b: whole-brain accuracy (violin, npc-selected) and accuracy-vs-npc curve
data_a = pd.DataFrame({
    "Accuracy": np.concatenate([anatomical_max_acc, hyper_max_acc]),
    "Method": ["Anatomically aligned"] * len(anatomical_max_acc) + ["Hyperaligned"] * len(hyper_max_acc),
})
t_stat, p_val = stats.ttest_rel(hyper_max_acc, anatomical_max_acc)
df = len(hyper_max_acc) - 1
p_text = f"p = {p_val:.3e}" if p_val < 0.001 else f"p = {p_val:.3f}"
stats_label = f"Paired t-test:\nt({df}) = {t_stat:.2f}\n{p_text}"
print(stats_label.replace(chr(10), ' | '))

pcs = np.arange(10, 301, 10)
hyper_tests = np.vstack([
    np.load(f"{root}classification/hyperaligned/{sid}_all_ridge_with_mask_unweighted_latest.npy")[:30]
    for sid in sids
])
anat_tests = np.vstack([
    np.load(f"{root}classification/anatomical/{sid}_with_mask_latest.npy")[:30]
    for sid in sids
])
hyper_mean = np.nanmean(hyper_tests, axis=0)
hyper_sem = np.nanstd(hyper_tests, axis=0, ddof=1) / np.sqrt(hyper_tests.shape[0])
anat_mean = np.nanmean(anat_tests, axis=0)
anat_sem = np.nanstd(anat_tests, axis=0, ddof=1) / np.sqrt(anat_tests.shape[0])

fig = plt.figure(figsize=(16, 13), dpi=300)
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.4], hspace=0.25, wspace=0.15)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[1, :])

violin_colors = ["#A6CEE3", "#FDBF6F"]
line_colors = ["#1F78B4", "#E31A1C"]
positions = [0, 0.4]
for method, v_color, l_color, pos in zip(data_a["Method"].unique(), violin_colors, line_colors, positions):
    subset = data_a[data_a["Method"] == method]["Accuracy"]
    parts = ax_a.violinplot(subset, positions=[pos], widths=0.3, showmeans=False, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(v_color)
        pc.set_edgecolor("black")
        pc.set_alpha(0.9)
    ax_a.scatter(np.random.normal(pos, 0.02, size=len(subset)), subset, color="black", s=25, alpha=0.7, zorder=3)
    mean_val = subset.mean()
    ax_a.hlines(mean_val, pos - 0.12, pos + 0.12, color=l_color, lw=3, zorder=4)
    ax_a.text(pos, mean_val + 0.002, f"{mean_val:.3f}", color=l_color, ha="center", va="bottom", fontsize=11, fontweight="bold")

chance_level = 1 / 900
ax_a.axhline(chance_level, color="gray", linestyle="--", lw=2, alpha=0.8)
ax_a.text(positions[-1] + 0.12, chance_level + 0.0002, "Chance level (1/900)", color="gray", fontsize=10, va="bottom")
ax_a.text(0.05, 0.95, stats_label, transform=ax_a.transAxes, fontsize=10, verticalalignment='top',
          horizontalalignment='left', bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='gray'))
ax_a.set_xticks(positions)
ax_a.set_xticklabels(["Anatomically-aligned", "Hyperaligned"], fontsize=13)
ax_a.set_ylabel("Accuracy", fontsize=15)
ax_a.set_title("Movie time point classification (Macaque)", fontsize=16, pad=15)
ax_a.grid(axis="y", linestyle="--", alpha=0.3)
sns.despine(ax=ax_a)

ax_b.plot(pcs, hyper_mean, label="Hyperaligned (mean)", linewidth=3, color="tab:orange")
ax_b.fill_between(pcs, hyper_mean - hyper_sem, hyper_mean + hyper_sem, alpha=0.25, color="tab:orange", label="Hyperaligned (± SEM)")
ax_b.plot(pcs, anat_mean, label="Anatomically-aligned (mean)", linewidth=3, color="tab:blue")
ax_b.fill_between(pcs, anat_mean - anat_sem, anat_mean + anat_sem, alpha=0.25, color="tab:blue", label="Anatomically-aligned (± SEM)")
ax_b.set_xlabel("# of PCs", fontsize=14, labelpad=8)
ax_b.set_ylabel("Test Accuracy", fontsize=14, labelpad=8)
ax_b.set_title("Average Accuracy vs PCs", fontsize=16, pad=15)
ax_b.set_xticks(pcs[::3])
ax_b.tick_params(axis='both', labelsize=12)
ax_b.legend(fontsize=11, loc="lower right")
ax_b.grid(alpha=0.3)
sns.despine(ax=ax_b)

ax_c.imshow(np.asarray(final_panel_c))
ax_c.axis('off')

ax_a.text(-0.10, 1.06, 'a', transform=ax_a.transAxes, fontsize=24, fontweight='bold', va='top', ha='right')
ax_b.text(-0.10, 1.06, 'b', transform=ax_b.transAxes, fontsize=24, fontweight='bold', va='top', ha='right')
ax_c.text(-0.01, 1.02, 'c', transform=ax_c.transAxes, fontsize=24, fontweight='bold', va='top', ha='right')

fig.savefig("combined_classification_results.png", dpi=300, bbox_inches='tight')
plt.show()

## Human (Supp. Fig. S3)

In [ ]:
dset = nb.MonkeyKingdom()
sids_h = dset.subjects

In [ ]:
hyper_max_acc_h = npc_selected_accuracy(
    sids_h,
    "classification/hyperaligned_train",
    "{exclude_sid}_all_human_ridge_unweighted_exclude_{sid}.npy",
    "classification/hyperaligned",
    "{sid}_all_human_ridge_unweighted_latest.npy",
)
anatomical_max_acc_h = npc_selected_accuracy(
    sids_h,
    "classification/anatomical_train",
    "{exclude_sid}_all_human_exclude_{sid}.npy",
    "classification/anatomical",
    "{sid}_human_latest.npy",
)

In [ ]:
# Panel c: human searchlight-average classification accuracy (onavg-ico32
# is already restricted to cortical vertices, so no separate mask expansion)
def searchlight_avg_map_human(out_tag):
    accs = [
        np.load(f"{root}classification/searchlight_analysis/{sid}_{out_tag}_timepoint_latest.npy")
        for sid in sids_h
    ]
    return np.mean(accs, axis=0)


anat_sl_h = searchlight_avg_map_human("anatomical_ridge_human")
hyper_sl_h = searchlight_avg_map_human("hyperalign_to_template_ridge_human_unweighted")

vmin, vmax = 0, 0.015
cmap = 'magma'
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
im1 = nb.plot(anat_sl_h, vmax=vmax, vmin=vmin, cmap=cmap, colorbar=False, title="Anatomically-aligned")
im3 = nb.plot(hyper_sl_h, vmax=vmax, vmin=vmin, cmap=cmap, colorbar=False, title="Hyperaligned")

fig_cb, ax_cb = plt.subplots(figsize=(11.3, 0.15), dpi=300)
fig_cb.subplots_adjust(0, 0, 1, 1)
mpl.colorbar.ColorbarBase(ax_cb, cmap=cmap, norm=norm, orientation='horizontal', ticks=[0, .005, .010, 0.015])
buf = io.BytesIO()
fig_cb.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
plt.close(fig_cb)
buf.seek(0)
cb_h = Image.open(buf).convert('RGBA')

top = nb.Image.hstack([im1, im3], padding=12)
panel_c_h = nb.Image.vstack([top, cb_h], padding=8)
png_bytes = panel_c_h._repr_png_()
panel_pil = Image.open(io.BytesIO(png_bytes)).convert('RGBA')
white_bg = Image.new("RGBA", panel_pil.size, "WHITE")
final_panel_c_h = Image.alpha_composite(white_bg, panel_pil).convert('RGB')

In [ ]:
# Panels a+b: human whole-brain accuracy (violin, npc-selected) and accuracy-vs-npc curve
data_a_h = pd.DataFrame({
    "Accuracy": np.concatenate([anatomical_max_acc_h, hyper_max_acc_h]),
    "Method": ["Anatomically aligned"] * len(anatomical_max_acc_h) + ["Hyperaligned"] * len(hyper_max_acc_h),
})
t_stat, p_val = stats.ttest_rel(hyper_max_acc_h, anatomical_max_acc_h)
df = len(hyper_max_acc_h) - 1
p_text = f"p = {p_val:.3e}" if p_val < 0.001 else f"p = {p_val:.3f}"
stats_label = f"Paired t-test:\nt({df}) = {t_stat:.2f}\n{p_text}"
print(stats_label.replace(chr(10), ' | '))

pcs = np.arange(10, 301, 10)
hyper_tests_h = np.vstack([
    np.load(f"{root}classification/hyperaligned/{sid}_all_human_ridge_unweighted_latest.npy")[:30]
    for sid in sids_h
])
anat_tests_h = np.vstack([
    np.load(f"{root}classification/anatomical/{sid}_human_latest.npy")[:30]
    for sid in sids_h
])
hyper_mean = np.nanmean(hyper_tests_h, axis=0)
hyper_sem = np.nanstd(hyper_tests_h, axis=0, ddof=1) / np.sqrt(hyper_tests_h.shape[0])
anat_mean = np.nanmean(anat_tests_h, axis=0)
anat_sem = np.nanstd(anat_tests_h, axis=0, ddof=1) / np.sqrt(anat_tests_h.shape[0])

fig = plt.figure(figsize=(16, 13), dpi=300)
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.4], hspace=0.25, wspace=0.15)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[1, :])

violin_colors = ["#A6CEE3", "#FDBF6F"]
line_colors = ["#1F78B4", "#E31A1C"]
positions = [0, 0.4]
for method, v_color, l_color, pos in zip(data_a_h["Method"].unique(), violin_colors, line_colors, positions):
    subset = data_a_h[data_a_h["Method"] == method]["Accuracy"]
    parts = ax_a.violinplot(subset, positions=[pos], widths=0.3, showmeans=False, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(v_color)
        pc.set_edgecolor("black")
        pc.set_alpha(0.9)
    ax_a.scatter(np.random.normal(pos, 0.02, size=len(subset)), subset, color="black", s=25, alpha=0.7, zorder=3)
    mean_val = subset.mean()
    ax_a.hlines(mean_val, pos - 0.12, pos + 0.12, color=l_color, lw=3, zorder=4)
    ax_a.text(pos, mean_val + 0.002, f"{mean_val:.3f}", color=l_color, ha="center", va="bottom", fontsize=11, fontweight="bold")

chance_level = 1 / 1800
ax_a.axhline(chance_level, color="gray", linestyle="--", lw=2, alpha=0.8)
ax_a.text(positions[-1] + 0.12, chance_level + 0.0002, "Chance level (1/1800)", color="gray", fontsize=10, va="bottom")
ax_a.text(0.05, 0.95, stats_label, transform=ax_a.transAxes, fontsize=10, verticalalignment='top',
          horizontalalignment='left', bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='gray'))
ax_a.set_xticks(positions)
ax_a.set_xticklabels(["Anatomically-aligned", "Hyperaligned"], fontsize=13)
ax_a.set_ylabel("Accuracy", fontsize=15)
ax_a.set_title("Movie time point classification (Human)", fontsize=16, pad=15)
ax_a.grid(axis="y", linestyle="--", alpha=0.3)
sns.despine(ax=ax_a)

ax_b.plot(pcs, hyper_mean, label="Hyperaligned (mean)", linewidth=3, color="tab:orange")
ax_b.fill_between(pcs, hyper_mean - hyper_sem, hyper_mean + hyper_sem, alpha=0.25, color="tab:orange", label="Hyperaligned (± SEM)")
ax_b.plot(pcs, anat_mean, label="Anatomically-aligned (mean)", linewidth=3, color="tab:blue")
ax_b.fill_between(pcs, anat_mean - anat_sem, anat_mean + anat_sem, alpha=0.25, color="tab:blue", label="Anatomically-aligned (± SEM)")
ax_b.set_xlabel("# of PCs", fontsize=14, labelpad=8)
ax_b.set_ylabel("Test Accuracy", fontsize=14, labelpad=8)
ax_b.set_title("Average Accuracy vs PCs", fontsize=16, pad=15)
ax_b.set_xticks(pcs[::3])
ax_b.tick_params(axis='both', labelsize=12)
ax_b.legend(fontsize=11, loc="lower right")
ax_b.grid(alpha=0.3)
sns.despine(ax=ax_b)

ax_c.imshow(np.asarray(final_panel_c_h))
ax_c.axis('off')

ax_a.text(-0.10, 1.06, 'a', transform=ax_a.transAxes, fontsize=24, fontweight='bold', va='top', ha='right')
ax_b.text(-0.10, 1.06, 'b', transform=ax_b.transAxes, fontsize=24, fontweight='bold', va='top', ha='right')
ax_c.text(-0.01, 1.02, 'c', transform=ax_c.transAxes, fontsize=24, fontweight='bold', va='top', ha='right')

fig.savefig("human_combined_classification_results.png", dpi=300, bbox_inches='tight')
plt.show()